In [ ]:
# Regular view: a saved SQL query straight from the gold table.
spark.sql('''
CREATE OR REPLACE VIEW adventure_works.gold.sales_kpis_category_quarter_vw AS
SELECT category, quarter, total_revenue, order_count, avg_order_value
FROM adventure_works.gold.sales_kpis_by_category_quarter
''')
print("regular view created")

In [ ]:
# Metric view: the KPI definitions live once in a governed Unity Catalog object.
spark.sql('''
CREATE OR REPLACE VIEW adventure_works.gold.sales_kpis_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: adventure_works.gold.sales_kpis_by_category_quarter
dimensions:
  - name: category
    expr: category
  - name: quarter
    expr: quarter
measures:
  - name: total_revenue
    expr: SUM(total_revenue)
  - name: order_count
    expr: SUM(order_count)
  - name: avg_order_value
    expr: SUM(total_revenue) / SUM(order_count)
$$
''')
print("metric view created")

In [ ]:
# The same question on both views: total revenue by category for the most recent quarter.
# 1) regular view
display(spark.sql('''
SELECT category, SUM(total_revenue) AS total_revenue
FROM adventure_works.gold.sales_kpis_category_quarter_vw
WHERE quarter = (SELECT MAX(quarter) FROM adventure_works.gold.sales_kpis_by_category_quarter)
GROUP BY category
ORDER BY total_revenue DESC
'''))

In [ ]:
# 2) metric view: measures are read with MEASURE(), both results should match
display(spark.sql('''
SELECT category, MEASURE(total_revenue) AS total_revenue
FROM adventure_works.gold.sales_kpis_metrics
WHERE quarter = (SELECT MAX(quarter) FROM adventure_works.gold.sales_kpis_by_category_quarter)
GROUP BY category
ORDER BY total_revenue DESC
'''))